In [ ]:
#Version: 1.0
import pandas as pd
import pandas_ta as ta
import yfinance as yf
import numpy as np
import Quadrant
import indicators
from concurrent.futures import ThreadPoolExecutor

# 1. 設定股票代碼和時間範圍
Ticker = [line.strip() for line in open('Mystocks.txt', encoding='utf-16') if line.strip()]

# 2. 實例化類別
ind_service = indicators.Indicators(period="2y", length=200)

# 3. 定義任務：下載資料 -> 計算指標 -> 取出最新一天數據
def task(t):
    # 1. 嚴格清理代碼字串：排除空行、包含空格或逗號的錯誤格式
    t = t.strip()
    if not t or ' ' in t or ',' in t:
        return None

    try:
        # 2. 下載後立刻呼叫 .copy()，強制切斷與 yfinance 內部快取的記憶體連結
        raw_df = yf.download(t, period=ind_service.period, progress=False)
        if raw_df.empty:
            return None
            
        # 建立完全獨立的副本進行後續運算
        df = raw_df.copy()
        
        # 寫入 ticker 並計算指標
        df['ticker'] = t
        df_analyzed = ind_service.get_indicators(df)
        
        # 提取最新數據
        latest_data = df_analyzed.iloc[-1].to_dict()
        latest_data['ticker'] = t 
        
        return latest_data

    except Exception as e:
        print(f"[{t}] 處理失敗: {e}")
        return None

# 4. 發動多執行緒
with ThreadPoolExecutor(max_workers=10) as executor:
    raw_results = list(executor.map(task, Ticker))

# 5. 清洗並轉成 DataFrame
cleaned_results = [r for r in raw_results if r is not None]
raw_data = pd.DataFrame(cleaned_results)

if not raw_data.empty:
    # 確保 ticker 成為 index
    raw_data.set_index('ticker', inplace=True)
    
    # 6. 分析象限
    analyzer = Quadrant.MarketQuadrantAnalyzer()
    final_result = analyzer.analyze_dataframe(raw_data)
    
    # 7. 存成 Excel
    final_result.to_excel("quadrant_results.xlsx")
    print("✅ 任務完成！")
else:
    print("❌ 沒抓到任何有效數據。")

In [ ]:
#Version: 1.1
import pandas as pd
import pandas_ta as ta
import yfinance as yf
import numpy as np
from Quadrant import Quadrant
import indicators
from concurrent.futures import ThreadPoolExecutor

# 1. 讀取並清理清單
Ticker = [line.strip() for line in open('Mystocks.txt', encoding='utf-16') if line.strip()]

# 2. 四象限篩選 (設定 3y 確保長天期指標能算出)
ind_service = indicators.Indicators(period="3y", length=200)

# ==========================================
# 階段一：多執行緒專職抓取 YoY (網路 I/O 密集)
# ==========================================
def fetch_yoy_only(t):
    try:
        tk = yf.Ticker(t)
        financials = tk.financials
        if financials is not None and 'Total Revenue' in financials.index:
            rev = financials.loc['Total Revenue']
            yoy = rev.pct_change(periods=-1)
            return {
                t: {
                    'yoy_now': yoy.iloc[0] if len(yoy) > 0 else np.nan,
                    'yoy_t1': yoy.iloc[1] if len(yoy) > 1 else np.nan,
                    'yoy_t2': yoy.iloc[2] if len(yoy) > 2 else np.nan
                }
            }
        return {t: {'yoy_now': np.nan, 'yoy_t1': np.nan, 'yoy_t2': np.nan}}
    except Exception:
        return {t: {'yoy_now': np.nan, 'yoy_t1': np.nan, 'yoy_t2': np.nan}}

print(f"啟動多執行緒抓取 {len(Ticker)} 檔 YoY 財報資料...")
with ThreadPoolExecutor(max_workers=10) as executor:
    yoy_results = list(executor.map(fetch_yoy_only, Ticker))

# 將結果整理成單一字典，方便後續查詢 { 'AAPL': {'yoy_now': 0.1...}, 'MSFT': {...} }
yoy_dict = {}
for res in yoy_results:
    yoy_dict.update(res)

# ==========================================
# 階段二：單執行緒循序處理價格與指標 (避免記憶體污染)
# ==========================================
print("啟動單執行緒計算歷史價格與技術指標...")
final_results = []

for t in Ticker:
    try:
        tk = yf.Ticker(t)
        raw_df = tk.history(period=ind_service.period)
        
        if raw_df.empty:
            continue
            
        df = raw_df.copy(deep=True)
        
        # 刻意不在此處加入 df['ticker'] = t
        # 這會觸發模組內 get_yoy 的防呆機制 (if not ticker: return df)，從而跳過耗時的重複下載
        df_analyzed = ind_service.get_indicators(df)
        
        # 提取最新一天數據
        latest_data = df_analyzed.iloc[-1].to_dict()
        
        # 補回股票代碼與階段一抓好的 YoY 數據
        latest_data['ticker'] = t
        latest_data.update(yoy_dict.get(t, {}))
        
        final_results.append(latest_data)
        
    except Exception as e:
        print(f"[{t}] 指標計算失敗: {e}")

# ==========================================
# 階段三：資料匯整與輸出
# ==========================================
raw_data = pd.DataFrame(final_results)

if not raw_data.empty:
    raw_data.set_index('ticker', inplace=True)
    
    # 分析象限
    analyzer = Quadrant.MarketQuadrantAnalyzer()
    final_result = analyzer.analyze_dataframe(raw_data)
    
    # 存成 Excel
    final_result.to_excel("quadrant_results.xlsx")
    print("✅ 任務完成！已儲存至 quadrant_results.xlsx")
else:
    print("❌ 沒抓到任何有效數據。")

# 3. 將目標股票的代碼放在 Mystocks-stage2.txt 裡，每行一檔，編碼為 UTF-16


In [ ]:
import pandas as pd
import yfinance as yf
from concurrent.futures import ThreadPoolExecutor
import numpy as np

# 1. 初始化 (實例化一次)
ind_service = indicators.Indicators(period="3y")

# 2. 準備清單
tickers = [line.strip() for line in open('Mystocks.txt', encoding='utf-16') if line.strip()]

# ---------------------------------------------------------
# 第一階段：多執行緒「只跑模組裡的 get_yoy」
# ---------------------------------------------------------
def thread_task(t):
    # 建立一個空容器給模組裝東西
    temp_df = pd.DataFrame(index=[0]) 
    try:
        # 呼叫模組：我們只用它的 get_yoy 方法
        res_df = ind_service.get_yoy(temp_df, ticker=t)
        return {t: res_df.iloc[0].to_dict()}
    except:
        return {t: {'yoy_now': np.nan, 'yoy_t1': np.nan, 'yoy_t2': np.nan}}

print("📡 正在多執行緒抓取 YoY 資料...")
with ThreadPoolExecutor(max_workers=10) as executor:
    results = list(executor.map(thread_task, tickers))

# 把結果壓平變成一個大字典，好讓後面查詢
yoy_master_dict = {k: v for d in results for k, v in d.items()}

# ---------------------------------------------------------
# 第二階段：單執行緒「跑剩下的指標」並組合
# ---------------------------------------------------------
print("📊 正在計算技術指標並組合資料...")
final_list = []

for t in tickers:
    try:
        # 抓取價格 (這裡很快，單執行緒才安全)
        df = yf.Ticker(t).history(period="3y")
        if df.empty: continue
        
        # 呼叫模組：跑 get_indicators (裡面包含 VIX、HV、MA 等)
        # 注意：我們「不」傳入 ticker 欄位，這樣模組內的 get_yoy 就會自動跳過
        df_analyzed = ind_service.get_indicators(df)
        
        # 拿取最後一列資料 (最新數據)
        latest_row = df_analyzed.iloc[-1].to_dict()
        
        # 關鍵：手工把階段一抓好的 yoy 字典塞進去
        latest_row['ticker'] = t
        latest_row.update(yoy_master_dict.get(t, {}))
        
        final_list.append(latest_row)
    except Exception as e:
        print(f"❌ {t} 計算出錯: {e}")

# ==========================================
# 第三階段：資料匯整、篩選與「分類」存檔
# ==========================================
raw_data = pd.DataFrame(final_list)

if not raw_data.empty:
    raw_data.set_index('ticker', inplace=True)
    
    # 6. 執行象限分析
    analyzer = Quadrant.MarketQuadrantAnalyzer()
    final_result = analyzer.analyze_dataframe(raw_data)
    
    # 7. 匯出 Excel：只保留 X_Score, Y_Score, Quadrant 並過濾 Q1 & Q4
    columns_to_keep = ['X_Score', 'Y_Score', 'Quadrant']
    mask_q14 = final_result['Quadrant'].isin([1, 4])
    excel_output = final_result[mask_q14][columns_to_keep]
    excel_output.to_excel("quadrant_results_filtered.xlsx")
    
    # 8. 分別提取第 1 象限與第 4 象限的 Tickers
    q1_tickers = final_result[final_result['Quadrant'] == 1].index.tolist()
    q4_tickers = final_result[final_result['Quadrant'] == 4].index.tolist()
    
    # 9. 分別存成兩個文字檔 (使用 utf-16 編碼)
    
    # 儲存 Q1 檔案
    with open('Mystocks-Q1.txt', 'w', encoding='utf-16') as f1:
        for ticker in q1_tickers:
            f1.write(f"{ticker}\n")
            
    # 儲存 Q4 檔案
    with open('Mystocks-Q4.txt', 'w', encoding='utf-16') as f4:
        for ticker in q4_tickers:
            f4.write(f"{ticker}\n")
            
    print(f"✅ 任務完成！")
    print(f"📈 第 1 象限 (Q1): 發現 {len(q1_tickers)} 檔股票 -> 已存至 Mystocks-Q1.txt")
    print(f"📉 第 4 象限 (Q4): 發現 {len(q4_tickers)} 檔股票 -> 已存至 Mystocks-Q4.txt")
    print(f"📊 總評分表已存至 quadrant_results_filtered.xlsx")

else:
    print("❌ 沒抓到任何有效數據。")

In [ ]:
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
import yfinance_fetcher
from indicators import Indicators
from Quadrant import MarketQuadrantAnalyzer

# 1. 實例化所有工具 (假設你已經定義好這些 Class)
# fetcher = FinanceFetcher() # 請取消註解並確保已定義
ind = Indicators()
ana = MarketQuadrantAnalyzer()

# --- 開始執行單檔測試 ---
#ticker = "1101.TW"
ticker = [line.strip() for line in open('Mystocks.txt', encoding='utf-16') if line.strip()]

# 2. 定義單次測試函數
def test_pipeline(ticker):
    print(f"--- 正在測試 {ticker} ---")
    try:
        # 步驟 A: 抓取資料 (請確保你的 fetcher 方法名稱正確)
        # 如果還沒寫好 fetcher，暫時用 yf.download(ticker, period="2y").reset_index() 代替測試
        fetcher=yfinance_fetcher.Yfinance_fetcher()
        df = fetcher.fetch(ticker, period="2y")
        
        if df is None or df.empty:
            print("❌ 抓取資料失敗：Dataframe 為空")
            return None
        
        # 步驟 B: 計算指標
        df_with_inds = ind.get_indicators(df)
        print(f"✅ 指標計算完成，欄位包含: {list(df_with_inds.columns)}")
        
        # 步驟 C: 象限分析
        df_final = ana.analyze_dataframe(df_with_inds)
        df_final = ana.attach_descriptions(df_final)
        
        # 顯示最新一筆結果
        latest = df_final.iloc[-1]
        print(f"\n最新狀態摘要 ({latest['ticker']}):")
        print(f"📅 日期: {latest.name if isinstance(latest.name, str) else 'Latest'}")
        print(f"📈 X 分數 (擴張): {latest['x_score']}")
        print(f"📉 Y 分數 (波動): {latest['y_score']}")
        print(f"🎯 所屬象限: {latest['quadrant']}")
        print(f"💡 策略建議: {latest['quadrant_strategy']}")
        
        return df_final
    except Exception as e:
        print(f"❌ 測試發生錯誤: {e}")
        import traceback
        traceback.print_exc()
        return None
    
result_df = test_pipeline(ticker)
result_df.to_excel("test_output.xlsx")


In [ ]:
import pandas as pd
import pandas_ta as ta
import yfinance as yf
import numpy as np
from indicators import Indicators
import yfinance_fetcher
import Quadrant

ticker = [line.strip() for line in open('Mystocks.txt', encoding='utf-16') if line.strip()]

def quadrant_analysis(ticker):
    ind=Indicators()
    fetcher = yfinance_fetcher.Yfinance_fetcher()
    ana = Quadrant.MarketQuadrantAnalyzer()
    df = fetcher.fetch(ticker, period="2y")
    df_ind = ind.get_indicators(df)
    df_final = ana.analyze_dataframe(df_ind)
    df_final = ana.attach_descriptions(df_final)
    return df_final

result_df = quadrant_analysis(ticker)
result_df.to_excel("test_output.xlsx")
display(result_df)

In [ ]:
import pandas as pd
import vectorbt as vbt
import numpy as np
from indicators import Indicators
import yfinance_fetcher
import Quadrant

# 1. 讀取標的
ticker_list = [line.strip() for line in open('Mystocks_s.txt', encoding='utf-16') if line.strip()]

def quadrant_analysis(ticker):
    ind = Indicators()
    fetcher = yfinance_fetcher.Yfinance_fetcher()
    ana = Quadrant.MarketQuadrantAnalyzer()
    
    # 抓取資料 (確保長度足以支撐 225 天暖機 + 2 年回測)
    df = fetcher.fetch(ticker, period="3y") 
    df_ind = ind.get_indicators(df)
    df_final = ana.analyze_dataframe(df_ind)
    df_final = ana.attach_descriptions(df_final)
    return df_final

# 準備存儲所有標的績效的清單
all_stats = []
portfolio_values = []  # + 新增一個清單，用來收集每檔股票的每日資金變化

for ticker in ticker_list:
    print(f"正在回測: {ticker}...")
    
    # 執行妳原有的分析模組
    df = quadrant_analysis(ticker)
    
    # --- 2. 定義進出場訊號 (向量化運算) ---
    
    # 進場：前一天 Q4 (x>0, y<=0) 且 當天 Q1 (x>0, y>=1)
    q4_prev = (df['x_score'].shift(1) > 0) & (df['y_score'].shift(1) <= 0)
    q3_prev = (df['x_score'].shift(1) < 0) & (df['y_score'].shift(1) <= 0)
    q1_curr = (df['x_score'] > 0) & ((df['y_score'] - df['y_score'].shift(1)) >= 1)
    entries = q4_prev & q1_curr | q3_prev & q1_curr
    
    # 出場：delta x > 1 連續兩天 OR x 轉負 (轉向收縮)
    dx = df['x_score'].diff()
    
    # 條件 1：動能衰退 (y 掉回 0 以下)
    decay_exit = (df['y_score'] < 0)
    # 條件 2：擴張停滯 (連續兩天 x 分數沒有成長)
    #stagnate_exit = (dx <= 0) & (dx.shift(1) <= 0)
    # 條件 3：趨勢反轉 (x 轉負)
    trend_exit = (df['x_score'] < 0)
    # 組合出場訊號
    #exits = decay_exit | stagnate_exit | trend_exit
    exits = decay_exit | trend_exit
    
    # --- 3. 處理暖機期與訊號清理 ---
    
    # 剔除前 225 天，避免指標未計算完成的偏誤
    entries[:225] = False
    exits[:225] = False

    # 加入這兩行來除錯：
    print(f"[{ticker}] 總資料筆數: {len(df)}")
    print(f"[{ticker}] 實際進場訊號數: {entries.sum()}, 實際出場訊號數: {exits.sum()}")
    
    # --- 4. 使用 vectorbt 執行回測 ---
    
    pf = vbt.Portfolio.from_signals(
        df['close'], 
        entries, 
        exits,
        fees=0.001,
        freq='1D', 
        init_cash=100000,
        # --- 加入以下這兩行 ---
        tp_stop=0.07,  # 7% 停利
        #sl_stop=0.05   # 5% 停損
    )
    
    # 收集統計數據
    stats = pf.stats()
    stats['Ticker'] = ticker
    all_stats.append(stats)
    
    # 如果想看個別標的的圖表，可以取消註解下一行
    #pf.plot().show()

    # 收集這檔股票的每日資金水位
    portfolio_values.append(pf.value())

'''
# 5. 彙整結果並輸出(Simplified)
final_perf_df = pd.concat(all_stats, axis=1).T
final_perf_df.to_excel("backtest_summary.xlsx")

print("\n--- 回測彙整報告 ---")
display(final_perf_df[['Ticker', 'Start Value', 'End Value', 'Total Return [%]', 'Max Drawdown [%]', 'Sharpe Ratio', 'Win Rate [%]']])
'''
# 5. ---彙整結果並輸出---
# 1. 輸出個別標的總表
final_perf_df = pd.concat(all_stats, axis=1).T
final_perf_df.to_excel("backtest_summary.xlsx")
print("\n--- 個別標的回測彙整 ---")
display(final_perf_df[['Ticker', 'Start Value', 'End Value', 'Total Return [%]', 'Max Drawdown [%]', 'Sharpe Ratio']])

# 2. 計算並輸出整體投資組合績效
# 將所有股票的每日資金加總，形成一條總權益曲線
total_equity = pd.concat(portfolio_values, axis=1).sum(axis=1)

# 計算每日的百分比報酬率，並剔除第一天的空值
overall_returns = total_equity.pct_change().dropna()

# 使用 vectorbt 的 returns 模組計算統計數據 (必須指定 freq='1D')
overall_stats = overall_returns.vbt.returns(freq='1D').stats()

print("\n--- 整體投資組合總績效 (All-in-One Portfolio) ---")
print(overall_stats)

In [ ]:
import pandas as pd
import vectorbt as vbt
import numpy as np
from indicators import Indicators
import yfinance_fetcher
import Quadrant

# 1. 讀取標的
ticker_list = [line.strip() for line in open('Mystocks.txt', encoding='utf-16') if line.strip()]

def quadrant_analysis(ticker):
    ind = Indicators()
    fetcher = yfinance_fetcher.Yfinance_fetcher()
    ana = Quadrant.MarketQuadrantAnalyzer()
    
    # 抓取資料 (確保長度足以支撐 225 天暖機 + 2 年回測)
    df = fetcher.fetch(ticker, period="3y") 
    df_ind = ind.get_indicators(df)
    df_final = ana.analyze_dataframe(df_ind)
    df_final = ana.attach_descriptions(df_final)
    return df_final

# 準備存儲所有標的績效的清單
all_stats = []
portfolio_values = []  # + 新增一個清單，用來收集每檔股票的每日資金變化

for ticker in ticker_list:
    #print(f"正在回測: {ticker}...")
    
    # 執行妳原有的分析模組
    df = quadrant_analysis(ticker)
    
    # --- 2. 定義進出場訊號 (向量化運算) ---
    
    # 進場：前一天 Q4 (x>0, y<=0) 且 當天 Q1 (x>0, y>=1)
    q4_prev = (df['x_score'].shift(1) > 0) & (df['y_score'].shift(1) <= 0)
    q1_curr = (df['x_score'] > 0) & ((df['y_score'] - df['y_score'].shift(1)) >= 1)
    entries = q4_prev & q1_curr
    
    # 出場：delta x > 1 連續兩天 OR x 轉負 (轉向收縮)
    dx = df['x_score'].diff()
    
    # 條件 1：動能衰退 (y 掉回 0 以下)
    #decay_exit = (df['y_score'] < 0)
    # 條件 2：擴張停滯 (連續兩天 x 分數沒有成長)
    stagnate_exit = (dx <= 0) & (dx.shift(1) <= 0)
    # 條件 3：趨勢反轉 (x 轉負)
    trend_exit = (df['x_score'] < 0)
    # 組合出場訊號
    #exits = decay_exit | stagnate_exit | trend_exit
    exits = stagnate_exit | trend_exit
    
    # --- 3. 處理暖機期與訊號清理 ---
    
    # 剔除前 225 天，避免指標未計算完成的偏誤
    entries[:225] = False
    exits[:225] = False

    # 加入這兩行來除錯：
    #print(f"[{ticker}] 總資料筆數: {len(df)}")
    #print(f"[{ticker}] 實際進場訊號數: {entries.sum()}, 實際出場訊號數: {exits.sum()}")
    
    # --- 4. 使用 vectorbt 執行回測 ---
    
    pf = vbt.Portfolio.from_signals(
        df['close'], 
        entries, 
        exits,
        fees=0.001,
        freq='1D', 
        init_cash=100000,
        # --- 加入以下這兩行 ---
        #tp_stop=0.07,  # 7% 停利
        #sl_stop=0.05   # 5% 停損
    )
    
    # 收集統計數據
    stats = pf.stats()
    stats['Ticker'] = ticker
    all_stats.append(stats)
    
    # 如果想看個別標的的圖表，可以取消註解下一行
    #pf.plot().show()

    # 收集這檔股票的每日資金水位
    portfolio_values.append(pf.value())

'''
# 5. 彙整結果並輸出(Simplified)
final_perf_df = pd.concat(all_stats, axis=1).T
final_perf_df.to_excel("backtest_summary.xlsx")

print("\n--- 回測彙整報告 ---")
display(final_perf_df[['Ticker', 'Start Value', 'End Value', 'Total Return [%]', 'Max Drawdown [%]', 'Sharpe Ratio', 'Win Rate [%]']])
'''
# 5. ---彙整結果並輸出---
# 1. 輸出個別標的總表
final_perf_df = pd.concat(all_stats, axis=1).T
final_perf_df.to_excel("backtest_summary.xlsx")
print("\n--- 個別標的回測彙整 ---")
display(final_perf_df[['Ticker', 'Start Value', 'End Value', 'Total Return [%]', 'Max Drawdown [%]', 'Sharpe Ratio']])

# 2. 計算並輸出整體投資組合績效
# 將所有股票的每日資金加總，形成一條總權益曲線
total_equity = pd.concat(portfolio_values, axis=1).sum(axis=1)

# 計算每日的百分比報酬率，並剔除第一天的空值
overall_returns = total_equity.pct_change().dropna()

# 使用 vectorbt 的 returns 模組計算統計數據 (必須指定 freq='1D')
overall_stats = overall_returns.vbt.returns(freq='1D').stats()

print("\n--- 整體投資組合總績效 (All-in-One Portfolio) ---")
print(overall_stats)

In [ ]:
import pandas as pd
import vectorbt as vbt
import numpy as np
from indicators import Indicators
import yfinance_fetcher
import Quadrant

# 1. 讀取標的
ticker_list = [line.strip() for line in open('Mystocks_s.txt', encoding='utf-16') if line.strip()]

def quadrant_analysis(ticker):
    ind = Indicators()
    fetcher = yfinance_fetcher.Yfinance_fetcher()
    ana = Quadrant.MarketQuadrantAnalyzer()
    
    # 抓取資料 (確保長度足以支撐 225 天暖機 + 2 年回測)
    df = fetcher.fetch(ticker, period="3y") 
    df_ind = ind.get_indicators(df)
    df_final = ana.analyze_dataframe(df_ind)
    df_final = ana.attach_descriptions(df_final)
    return df_final

# 準備存儲所有標的績效的清單
all_stats = []
portfolio_values = []  # + 新增一個清單，用來收集每檔股票的每日資金變化

for ticker in ticker_list:
    print(f"正在回測: {ticker}...")
    
    # 執行妳原有的分析模組
    df = quadrant_analysis(ticker)
    
    # --- 2. 定義進出場訊號 (向量化運算) ---
    
    # 進場
    q4_prev = ((df['x_score'].shift(1) > 0) & (df['y_score'].shift(1) < 0)&
               (df['x_score'].shift(2) > 0) & (df['y_score'].shift(2) < 0)&
               (df['x_score'].shift(3) > 0) & (df['y_score'].shift(3) < 0)&
               (df['x_score'].shift(4) > 0) & (df['y_score'].shift(4) < 0)&
               (df['x_score'].shift(5) > 0) & (df['y_score'].shift(5) < 0))
    q1_prev = ((df['x_score'].shift(1) > 0) & (df['y_score'].shift(1) < 0)&
               (df['x_score'].shift(2) > 0) & (df['y_score'].shift(2) < 0)&
               (df['x_score'].shift(3) <= 0) & (df['y_score'].shift(3) <= 0)&
               (df['x_score'].shift(4) <= 0) & (df['y_score'].shift(4) <= 0))
    q4_curr = (df['x_score'] > 0) & (df['y_score'] <= 0)
    q1_curr = (df['x_score'] > 0) & (df['y_score'] >= 0)
    entries = q4_prev & q4_curr | q1_prev & q1_curr
    
    # 出場：Quadrant1
    # 止損：開始收縮
    trend_exit_q1 = ((df['x_score']<df['x_score'].shift(1))&
                     (df['x_score'].shift(1)<df['x_score'].shift(2))&
                     df['x_score'] < 0) 
    # 止盈：動能收縮
    profit_exit_q1 = ((df['y_score']<df['y_score'].shift(5))&
                      (df['y_score']<df['y_score'].shift(10)))
    # 出場：Quadrant4
    # 止損：開始收縮
    trend_exit_q4 = ((df['x_score']<df['x_score'].shift(5))&
                     (df['x_score']<df['x_score'].shift(10))&
                     df['x_score'] < 0)
    # 止盈
    profit_exit_q4 = ((df['y_score'] > 0) & (df['y_score'] - df['y_score'].shift(1) >= 1))
    # 組合出場訊號：delta x > 1 連續兩天 OR x 轉負 (轉向收縮)
    dx = df['x_score'].diff()
    stagnate_exit = (dx < 0) & (dx.shift(1) < 0)

    exits = trend_exit_q1 | trend_exit_q4 | profit_exit_q4 | stagnate_exit
    
    # --- 3. 處理暖機期與訊號清理 ---
    
    # 剔除前 225 天，避免指標未計算完成的偏誤
    entries[:225] = False
    exits[:225] = False

    # 加入這兩行來除錯：
    print(f"[{ticker}] 總資料筆數: {len(df)}")
    print(f"[{ticker}] 實際進場訊號數: {entries.sum()}, 實際出場訊號數: {exits.sum()}")
    
    # --- 4. 使用 vectorbt 執行回測 ---
    
    pf = vbt.Portfolio.from_signals(
        df['close'], 
        entries, 
        exits,
        fees=0.001,
        freq='1D', 
        init_cash=100000,
        tp_stop=0.1,  # 停利
        #sl_stop=0.1   # 停損
    )
    
    # 收集統計數據
    stats = pf.stats()
    stats['Ticker'] = ticker
    all_stats.append(stats)
    
    # 如果想看個別標的的圖表，可以取消註解下一行
    #pf.plot().show()

    # 收集這檔股票的每日資金水位
    portfolio_values.append(pf.value())


# 5. ---彙整結果並輸出---
# 1. 輸出個別標的總表
final_perf_df = pd.concat(all_stats, axis=1).T
final_perf_df.to_excel("backtest_summary.xlsx")
print("\n--- 個別標的回測彙整 ---")
display(final_perf_df[['Ticker', 'Start Value', 'End Value', 'Total Return [%]', 'Max Drawdown [%]', 'Sharpe Ratio']])

# 2. 計算並輸出整體投資組合績效
# 將所有股票的每日資金加總，形成一條總權益曲線
total_equity = pd.concat(portfolio_values, axis=1).sum(axis=1)

# 計算每日的百分比報酬率，並剔除第一天的空值
overall_returns = total_equity.pct_change().dropna()

# 使用 vectorbt 的 returns 模組計算統計數據 (必須指定 freq='1D')
overall_stats = overall_returns.vbt.returns(freq='1D').stats()

print("\n--- 整體投資組合總績效 (All-in-One Portfolio) ---")
print(overall_stats)

In [ ]:
import pandas as pd
import vectorbt as vbt
import numpy as np
from indicators import Indicators
import yfinance_fetcher
import Quadrant

# 1. 讀取標的
ticker_list = [line.strip() for line in open('Mystocks_s.txt', encoding='utf-16') if line.strip()]

def quadrant_analysis(ticker):
    ind = Indicators()
    fetcher = yfinance_fetcher.Yfinance_fetcher()
    ana = Quadrant.MarketQuadrantAnalyzer()
    
    # 抓取資料 (確保長度足以支撐 225 天暖機 + 2 年回測)
    df = fetcher.fetch(ticker, period="3y") 
    df_ind = ind.get_indicators(df)
    df_final = ana.analyze_dataframe(df_ind)
    df_final = ana.attach_descriptions(df_final)
    return df_final

# 準備存儲所有標的績效的清單
all_stats = []
portfolio_values = []  # + 新增一個清單，用來收集每檔股票的每日資金變化

for ticker in ticker_list:
    print(f"正在回測: {ticker}...")
    
    # 執行妳原有的分析模組
    df = quadrant_analysis(ticker)
    
    # --- 2. 定義進出場訊號 (向量化運算) ---
    
    # 進場
    q4_prev = ((df['x_score'].shift(1) > 0) & (df['y_score'].shift(1) < 0)&
               (df['x_score'].shift(2) > 0) & (df['y_score'].shift(2) < 0)&
               (df['x_score'].shift(3) > 0) & (df['y_score'].shift(3) < 0)&
               (df['x_score'].shift(4) > 0) & (df['y_score'].shift(4) < 0)&
               (df['x_score'].shift(5) > 0) & (df['y_score'].shift(5) < 0))
    q1_prev = ((df['x_score'].shift(1) > 0) & (df['y_score'].shift(1) < 0)&
               (df['x_score'].shift(2) > 0) & (df['y_score'].shift(2) < 0))
    q4_curr = (df['x_score'] > 0) & (df['y_score'] <= 0)
    q1_curr = (df['x_score'] > 0) & (df['y_score'] >= 0)
    entries = q4_prev & q4_curr | q1_prev & q1_curr
    
    # 出場：Quadrant1
    # 止損：開始收縮
    trend_exit_q1 = ((df['x_score']<df['x_score'].shift(1))&
                     (df['x_score'].shift(1)<df['x_score'].shift(2))&
                     df['x_score'] < 0) 
    # 止盈：動能收縮
    profit_exit_q1 = ((df['y_score']<df['y_score'].shift(5))&
                      (df['y_score']<df['y_score'].shift(10)))
    # 出場：Quadrant4
    # 止損：開始收縮
    trend_exit_q4 = ((df['x_score']<df['x_score'].shift(5))&
                     (df['x_score']<df['x_score'].shift(10))&
                     df['x_score'] < 0)
    # 止盈
    profit_exit_q4 = ((df['y_score'] > 0) & (df['y_score'] - df['y_score'].shift(1) >= 1))
    # 組合出場訊號：delta x > 1 連續兩天 OR x 轉負 (轉向收縮)
    dx = df['x_score'].diff()
    stagnate_exit = (dx < 0) & (dx.shift(1) < 0)

    exits = trend_exit_q1 | trend_exit_q4 | profit_exit_q4 | stagnate_exit
    
    # --- 3. 處理暖機期與訊號清理 ---
    
    # 剔除前 225 天，避免指標未計算完成的偏誤
    entries[:225] = False
    exits[:225] = False

    # 加入這兩行來除錯：
    print(f"[{ticker}] 總資料筆數: {len(df)}")
    print(f"[{ticker}] 實際進場訊號數: {entries.sum()}, 實際出場訊號數: {exits.sum()}")
    
    # --- 4. 使用 vectorbt 執行回測 ---
    
    pf = vbt.Portfolio.from_signals(
        df['close'], 
        entries, 
        exits,
        fees=0.001,
        freq='1D', 
        init_cash=100000,
        tp_stop=0.1,  # 停利
        #sl_stop=0.1   # 停損
    )
    
    # 收集統計數據
    stats = pf.stats()
    stats['Ticker'] = ticker
    all_stats.append(stats)
    
    # 如果想看個別標的的圖表，可以取消註解下一行
    #pf.plot().show()

    # 收集這檔股票的每日資金水位
    portfolio_values.append(pf.value())


# 5. ---彙整結果並輸出---
# 1. 輸出個別標的總表
final_perf_df = pd.concat(all_stats, axis=1).T
final_perf_df.to_excel("backtest_summary.xlsx")
print("\n--- 個別標的回測彙整 ---")
display(final_perf_df[['Ticker', 'Start Value', 'End Value', 'Total Return [%]', 'Max Drawdown [%]', 'Sharpe Ratio']])

# 2. 計算並輸出整體投資組合績效
# 將所有股票的每日資金加總，形成一條總權益曲線
total_equity = pd.concat(portfolio_values, axis=1).sum(axis=1)

# 計算每日的百分比報酬率，並剔除第一天的空值
overall_returns = total_equity.pct_change().dropna()

# 使用 vectorbt 的 returns 模組計算統計數據 (必須指定 freq='1D')
overall_stats = overall_returns.vbt.returns(freq='1D').stats()

print("\n--- 整體投資組合總績效 (All-in-One Portfolio) ---")
print(overall_stats)

In [ ]:
#放寬進出場條件，看看能不能增加交易次數
import pandas as pd
import vectorbt as vbt
import numpy as np
from indicators import Indicators
import yfinance_fetcher
import Quadrant

# 1. 讀取標的
ticker_list = [line.strip() for line in open('Mystocks_s.txt', encoding='utf-16') if line.strip()]

def quadrant_analysis(ticker):
    ind = Indicators()
    fetcher = yfinance_fetcher.Yfinance_fetcher()
    ana = Quadrant.MarketQuadrantAnalyzer()
    
    # 抓取資料 (確保長度足以支撐 225 天暖機 + 2 年回測)
    df = fetcher.fetch(ticker, period="3y") 
    df_ind = ind.get_indicators(df)
    df_final = ana.analyze_dataframe(df_ind)
    df_final = ana.attach_descriptions(df_final)
    return df_final

# 準備存儲所有標的績效的清單
all_stats = []
portfolio_values = []  # + 新增一個清單，用來收集每檔股票的每日資金變化

for ticker in ticker_list:
    print(f"正在回測: {ticker}...")
    
    # 執行妳原有的分析模組
    df = quadrant_analysis(ticker)
    
    # --- 2. 定義進出場訊號 (向量化運算) ---
    
    # 進場
    q4_prev = ((df['x_score'].shift(1) > 0) & (df['y_score'].shift(1) < 0)&
               (df['x_score'].shift(2) > 0) & (df['y_score'].shift(2) < 0)&
               (df['x_score'].shift(3) > 0) & (df['y_score'].shift(3) < 0))
    q1_prev = ((df['x_score'].shift(1) > 0) & (df['y_score'].shift(1) < 0)&
               (df['x_score'].shift(2) > 0) & (df['y_score'].shift(2) < 0)&
               (df['x_score'].shift(3) <= 0) & (df['y_score'].shift(3) <= 0))
    q4_curr = (df['x_score'] > 0) & (df['y_score'] <= 0)
    q1_curr = (df['x_score'] > 0) & (df['y_score'] >= 0)
    entries = q4_prev & q4_curr | q1_prev & q1_curr
    
    # 出場：Quadrant1
    # 止損：開始收縮
    trend_exit_q1 = ((df['x_score']<df['x_score'].shift(1))&
                     (df['x_score'].shift(1)<df['x_score'].shift(2))&
                     df['x_score'] < 0) 
    # 止盈：動能收縮
    profit_exit_q1 = ((df['y_score']<df['y_score'].shift(5))&
                      (df['y_score']<df['y_score'].shift(10)))
    # 出場：Quadrant4
    # 止損：開始收縮
    trend_exit_q4 = ((df['x_score']<df['x_score'].shift(5))&
                     (df['x_score']<df['x_score'].shift(10))&
                     df['x_score'] < 0)
    # 止盈
    profit_exit_q4 = ((df['y_score'] > 0) & (df['y_score'] - df['y_score'].shift(1) >= 1))
    # 組合出場訊號：delta x > 1 連續兩天 OR x 轉負 (轉向收縮)
    dx = df['x_score'].diff()
    stagnate_exit = (dx < 0) & (dx.shift(1) < 0)
    # 象限出場
    quadrant_exit_q2 = (df['x_score'] < 0) & (df['y_score'] > 0) & (df['x_score'].shift(1) < 0) & (df['y_score'].shift(1) > 0)
    quadrant_exit_q3 = (df['x_score'] < 0) & (df['y_score'] < 0) & (df['x_score'].shift(1) < 0) & (df['y_score'].shift(1) < 0)

    exits = trend_exit_q1 | trend_exit_q4 | profit_exit_q4 | stagnate_exit | quadrant_exit_q3 | quadrant_exit_q2
    
    # --- 3. 處理暖機期與訊號清理 ---
    
    # 剔除前 225 天，避免指標未計算完成的偏誤
    entries[:225] = False
    exits[:225] = False

    # 加入這兩行來除錯：
    print(f"[{ticker}] 總資料筆數: {len(df)}")
    print(f"[{ticker}] 實際進場訊號數: {entries.sum()}, 實際出場訊號數: {exits.sum()}")
    
    # --- 4. 使用 vectorbt 執行回測 ---
    
    pf = vbt.Portfolio.from_signals(
        df['close'], 
        entries, 
        exits,
        fees=0.001425,   # 手續費 (0.1425%)
        freq='1D', 
        init_cash=100000,       
        slippage=0.001,
        tp_stop=0.1  # 停利
        #sl_stop=0.1   # 停損
    )
    
    # 收集統計數據
    stats = pf.stats()
    stats['Ticker'] = ticker
    all_stats.append(stats)
    
    # 如果想看個別標的的圖表，可以取消註解下一行
    #pf.plot().show()

    # 收集這檔股票的每日資金水位
    portfolio_values.append(pf.value())


# 5. ---彙整結果並輸出---
# 1. 輸出個別標的總表
final_perf_df = pd.concat(all_stats, axis=1).T
final_perf_df.to_excel("backtest_summary.xlsx")
print("\n--- 個別標的回測彙整 ---")
display(final_perf_df[['Ticker', 'Start Value', 'End Value', 'Total Return [%]', 'Max Drawdown [%]', 'Sharpe Ratio']])

# 2. 計算並輸出整體投資組合績效
# 將所有股票的每日資金加總，形成一條總權益曲線
total_equity = pd.concat(portfolio_values, axis=1).sum(axis=1)

# 計算每日的百分比報酬率，並剔除第一天的空值
overall_returns = total_equity.pct_change().dropna()

# 使用 vectorbt 的 returns 模組計算統計數據 (必須指定 freq='1D')
overall_stats = overall_returns.vbt.returns(freq='1D').stats()

print("\n--- 整體投資組合總績效 (All-in-One Portfolio) ---")
print(overall_stats)

# 繪製並顯示資金曲線
# 使用 vbt 的繪圖功能，這會產生一個互動式的 Plotly 圖表
fig = total_equity.vbt.plot(
    trace_kwargs=dict(name='Total Equity', line=dict(color='blue')))

overall_returns.vbt.returns(freq='1D').plot().show()

# 設定圖表標題
fig.update_layout(title_text='整體投資組合資金曲線', xaxis_title='日期', yaxis_title='總價值')

# 顯示圖表
fig.show()